# 09 - Phase 3: second model (LLaVA-1.5-7B)
**Question.** Are the main findings (number tags help counting, landmarks do not help) specific to Qwen2-VL-7B, or do they also
show up with a different model? We run the same 8 variants on **LLaVA-1.5-7B** (4-bit, greedy) and compare it with Qwen.

| | |
|---|---|
| **Input** | `data/dataset.json` (needs the photos), Qwen answers in `results/Qwen_Qwen2-VL-7B-Instruct/raw_outputs.jsonl` |
| **Output** | `results/llava-hf_llava-1.5-7b-hf/raw_outputs.jsonl` + metrics CSVs, `cross_model_accuracy.csv` |
| **Needs** | GPU (Colab T4) |

Note: LLaVA-1.5 resizes every image to 336x336 px, so small faces in crowded photos lose detail - mention this when you compare.

In [ ]:
# Setup: find the repo root so `from src import ...` works (run this notebook from inside the repo).
import sys
from pathlib import Path
ROOT = next((p for p in [Path.cwd(), Path.cwd().parent] if (p / "src").is_dir()), None)
assert ROOT is not None, "Run from inside the repo (the folder containing src/). In Colab: upload the repo zip, unzip it and %cd into it."
sys.path.insert(0, str(ROOT))
from src import config, utils
print("repo root:", ROOT)

In [ ]:
# Colab only
!pip -q install -U transformers accelerate bitsandbytes pillow tqdm

In [ ]:
from src import models
DATA = utils.load_json(config.DATASET_JSON)
models.run_experiment(DATA, "llava", config.VARIANTS)

In [ ]:
import pandas as pd
from src import evaluation, labeling, plots
VARIANTS = config.VARIANTS
RES_L = config.results_dir("llava")
res_l = evaluation.evaluate(DATA, utils.load_raw_outputs(RES_L / "raw_outputs.jsonl"), VARIANTS)
res_q = evaluation.evaluate(DATA, utils.load_raw_outputs(config.results_dir("qwen") / "raw_outputs.jsonl"), VARIANTS)

pd.options.display.float_format = "{:.3f}".format
print("LLaVA-1.5-7B accuracy (per face):"); print(evaluation.wide(res_l["M"], "accuracy", VARIANTS).to_string(index=False))
print("\nLLaVA face-count accuracy:"); print(evaluation.wide(res_l["M"], "face_count", VARIANTS).to_string(index=False))
print("\nLLaVA accuracy difference vs plain (95% CI):"); print(res_l["ci"].assign(variant=res_l["ci"].variant.map(config.VARIANT_TITLES)).to_string(index=False))
print(f"\nLLaVA majority baseline: always '{res_l['maj_class']}' = {res_l['maj']:.1%}")
for v, top, share in res_l["top_answers"]:
    flag = "   <-- COLLAPSE" if v in res_l["collapsed"] else ""
    print(f"{config.VARIANT_TITLES[v]:45s} most common answer: {top} ({share:.0%}){flag}")
res_l["M"].to_csv(RES_L / "metrics.csv", index=False)
res_l["faces"].to_csv(RES_L / "per_face_predictions.csv", index=False)
res_l["ci"].to_csv(RES_L / "bootstrap_vs_baseline.csv", index=False)
res_l["per_class"].to_csv(RES_L / "per_class_recall.csv")
(RES_L / "results_summary.md").write_text(evaluation.summary_markdown(res_l, VARIANTS, RES_L.name))

In [ ]:
# Side by side with Qwen (overall accuracy %, and face-count accuracy %)
pd.options.display.float_format = "{:.1f}".format
models_M = {"Qwen2-VL-7B": res_q["M"], "LLaVA-1.5-7B": res_l["M"]}
ACC = plots.cross_model_table(models_M, VARIANTS, "accuracy")
CNT = plots.cross_model_table(models_M, VARIANTS, "face_count")
print("Accuracy %"); print(ACC.to_string(index=False)); print("\nFace-count accuracy %"); print(CNT.to_string(index=False))
ACC.to_csv(RES_L / "cross_model_accuracy.csv", index=False)

import matplotlib.pyplot as plt, numpy as np
fig, ax = plt.subplots(figsize=(13, 4.5)); w = 0.38; x = np.arange(len(VARIANTS))
ax.bar(x - w / 2, ACC["Qwen2-VL-7B"], w, label="Qwen2-VL-7B"); ax.bar(x + w / 2, ACC["LLaVA-1.5-7B"], w, label="LLaVA-1.5-7B")
ax.set_xticks(x); ax.set_xticklabels([v.split("_")[0] for v in VARIANTS]); ax.set_ylabel("Accuracy %"); ax.set_ylim(0, 100)
ax.axhline(res_q["maj"] * 100, color="gray", ls="--", lw=1, label=f"always '{res_q['maj_class']}'")
ax.set_title("Per-face accuracy by visual prompt: two models"); ax.legend(); plt.tight_layout()
plt.savefig(RES_L / "figures" / "cross_model_accuracy.png", dpi=130); plt.show()

## How to read the result (write your own conclusion)
- Do the **same patterns** hold (numbers help counting, landmarks do not help accuracy)? Then the finding is not specific to Qwen.
- Does LLaVA collapse to one answer (e.g. almost always "neutral" or "happy")? Then its accuracy is not meaningful - say so.
- Different patterns -> the paper's SoV gain may depend on the model (the paper used GPT-4V).